# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanhGiauTen/flyrankAI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook compares learned ranking scores with the frozen Week-4 low-CTR baseline on the same eligible rows, client holdout, and top-K metrics.

## 1. Method choice and why

The decision is a ranking problem, supported by a binary teaching proxy (`trend_direction == 'down'`). I start with logistic regression because its direction is readable, then compare a shallow decision tree and a constrained random forest because nonlinear interactions between position, volume, age, and engagement are plausible. Model probabilities become ranking scores; complexity is accepted only if it beats the frozen rule at Precision@20/50 on unseen clients. The result is current-window decision support, not a future forecast.

In [1]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
DATA_URL = 'https://raw.githubusercontent.com/KhanhGiauTen/flyrankAI/main/data/raw/content_refresh_anonymized.csv'
local_candidates = [Path('data/raw/content_refresh_anonymized.csv'), Path('../../data/raw/content_refresh_anonymized.csv')]
data_source = next((path for path in local_candidates if path.exists()), DATA_URL)
raw = pd.read_csv(data_source)
print({'python': platform.python_version(), 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__, 'seed': RANDOM_STATE})


{'python': '3.11.9', 'pandas': '2.3.3', 'scikit_learn': '1.8.0', 'seed': 42}


## 2. Split design

Pages from the same client share portfolio and measurement patterns, so a random row split would make the test artificially familiar. I hold out about 20% of pseudonymized clients using a fixed seed. The split is created before model fitting and before estimating the baseline's position-band CTR medians. Both the rule and every model are then evaluated on exactly the same held-out rows. IDs are used only for grouping and audit output, never as features.

In [2]:
frame = raw[(raw['impressions_90d'] >= 500) & (raw['avg_position'] > 0) & (raw['avg_position'] <= 50)].copy()
frame['is_declining_proxy'] = frame['trend_direction'].eq('down').astype('int8')
frame['position_band'] = pd.cut(
    frame['avg_position'], [0, 3, 10, 20, 50],
    labels=['top_3', 'page_1', 'striking', 'page_3_5'], include_lowest=True
)
clients = frame['client_id'].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
test_client_count = max(1, round(len(clients) * 0.20))
test_clients = set(rng.permutation(clients)[:test_client_count])
train_mask = ~frame['client_id'].isin(test_clients)
test_mask = ~train_mask
train = frame.loc[train_mask].copy()
test = frame.loc[test_mask].copy()
assert set(train['client_id']).isdisjoint(set(test['client_id']))
assert train['is_declining_proxy'].nunique() == test['is_declining_proxy'].nunique() == 2

train_position_medians = train.groupby('position_band', observed=True)['ctr'].median()
for split in (train, test):
    split['expected_ctr'] = split['position_band'].map(train_position_medians).astype(float)
    split['baseline_score'] = np.log1p(split['impressions_90d']) * (split['expected_ctr'] - split['ctr']).clip(lower=0)

split_summary = pd.DataFrame({
    'rows': [len(train), len(test)],
    'clients': [train.client_id.nunique(), test.client_id.nunique()],
    'proxy_positive_rate': [train.is_declining_proxy.mean(), test.is_declining_proxy.mean()],
}, index=['train', 'test'])
split_summary.style.format({'proxy_positive_rate': '{:.1%}'})


,rows,clients,proxy_positive_rate
train,12759,22,60.0%
test,3578,6,61.1%


## 3. Train + compare against the frozen baseline

Numeric missing values are median-imputed with explicit missingness indicators; categorical values are imputed and one-hot encoded. Direct answer fields (`trend_direction`, `trend_pct`, and recent/previous 30-day comparison inputs), IDs, provider/model names, and the proxy label are excluded. The comparison table includes the held-out base rate, Week-4 rule, and all models on identical rows.

In [3]:
numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct'
]
categorical_features = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier'
]
prohibited = {
    'content_id', 'client_id', 'trend_direction', 'trend_pct',
    'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d',
    'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d',
    'is_declining_proxy', 'provider_used', 'model_used'
}
assert set(numeric_features + categorical_features).isdisjoint(prohibited)

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), categorical_features),
])
models = {
    'logistic_regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'decision_tree': DecisionTreeClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=25,
        class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE
    ),
}
X_train = train[numeric_features + categorical_features]
X_test = test[numeric_features + categorical_features]
y_train = train['is_declining_proxy']
y_test = test['is_declining_proxy']

def precision_at_k(y_true, scores, k):
    audit = pd.DataFrame({'y': np.asarray(y_true), 'score': np.asarray(scores)})
    return float(audit.nlargest(min(k, len(audit)), 'score')['y'].mean())

def evaluate(scores):
    return {
        'precision_at_20': precision_at_k(y_test, scores, 20),
        'precision_at_50': precision_at_k(y_test, scores, 50),
        'average_precision': float(average_precision_score(y_test, scores)),
        'roc_auc': float(roc_auc_score(y_test, scores)),
    }

fitted = {}
score_vectors = {'base_rate': np.repeat(y_test.mean(), len(y_test)), 'week4_rule': test['baseline_score'].to_numpy()}
for name, estimator in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    score_vectors[name] = pipe.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({name: evaluate(scores) for name, scores in score_vectors.items()}).T
best_model_name = comparison.loc[list(models)].sort_values(
    ['precision_at_50', 'average_precision', 'roc_auc'], ascending=False
).index[0]
best_model = fitted[best_model_name]
best_scores = score_vectors[best_model_name]

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/content_refresh_anonymized.csv').exists()), Path.cwd())
output_dir = repo_root / 'work/outputs'
output_dir.mkdir(parents=True, exist_ok=True)
receipt = {
    'seed': RANDOM_STATE,
    'split': 'client_holdout',
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'test_clients': int(test.client_id.nunique()),
    'test_proxy_base_rate': float(y_test.mean()),
    'best_model': best_model_name,
    'comparison': json.loads(comparison.reset_index(names='method').to_json(orient='records')),
    'excluded_direct_answer_fields': sorted(prohibited),
}
(output_dir / 'model_metrics.json').write_text(json.dumps(receipt, indent=2), encoding='utf-8')
print('Selected model:', best_model_name)
comparison.style.format('{:.3f}').highlight_max(axis=0, color='#d9ead3')


Selected model: random_forest


,precision_at_20,precision_at_50,average_precision,roc_auc
base_rate,0.650,0.580,0.611,0.500
week4_rule,0.850,0.840,0.695,0.618
logistic_regression,0.800,0.760,0.709,0.617
decision_tree,0.650,0.820,0.678,0.614
random_forest,0.850,0.840,0.709,0.620


## 4. Errors and interpretation

I inspect feature importance only after the held-out comparison, and treat it as model reliance rather than causal effect. I also show concrete high-confidence false positives and low-confidence false negatives. A false positive near the queue top wastes review capacity; a false negative hides a potentially declining page. Error rates by content type reveal whether one overall metric masks a weak subgroup.

In [4]:
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
estimator = best_model.named_steps['model']
if hasattr(estimator, 'feature_importances_'):
    importance_values = estimator.feature_importances_
else:
    importance_values = np.abs(estimator.coef_[0])
importance = (
    pd.DataFrame({'feature': feature_names, 'importance': importance_values})
    .sort_values('importance', ascending=False)
    .head(12)
)
print('Top model reliance signals (not causal effects):')
display(importance)

audit = test[['content_id', 'content_type', 'impressions_90d', 'avg_position', 'ctr', 'is_declining_proxy']].copy()
audit['model_score'] = best_scores
audit['predicted_class'] = (audit['model_score'] >= 0.5).astype('int8')
audit['is_error'] = audit['predicted_class'].ne(audit['is_declining_proxy'])
false_positives = audit[(audit.predicted_class == 1) & (audit.is_declining_proxy == 0)].nlargest(3, 'model_score')
false_negatives = audit[(audit.predicted_class == 0) & (audit.is_declining_proxy == 1)].nsmallest(3, 'model_score')
mistakes = pd.concat([
    false_positives.assign(error_type='false_positive'),
    false_negatives.assign(error_type='false_negative'),
])[['error_type', 'content_id', 'content_type', 'impressions_90d', 'avg_position', 'ctr', 'model_score']]
print('Concrete difficult cases:')
display(mistakes)

error_by_type = audit.groupby('content_type', dropna=False).agg(
    n=('content_id', 'size'),
    error_rate=('is_error', 'mean'),
    proxy_rate=('is_declining_proxy', 'mean'),
).sort_values('error_rate', ascending=False)
print('Errors by content type:')
error_by_type.style.format({'error_rate': '{:.1%}', 'proxy_rate': '{:.1%}'})


Top model reliance signals (not causal effects):


,feature,importance
14,num__content_age_days,0.093329
19,num__scroll_rate,0.085582
17,num__avg_position,0.077640
6,num__clicks_90d,0.053997
12,num__days_with_impressions,0.046587
3,num__word_count,0.045996
16,num__ctr,0.044351
4,num__char_count,0.041740
5,num__impressions_90d,0.037302
8,num__users_90d,0.036521


Concrete difficult cases:


,error_type,content_id,content_type,impressions_90d,avg_position,ctr,model_score
23973,false_positive,content_be7555890f38,keyword article,5266,40.8,0.09,0.836364
1079,false_positive,content_a467a30832b8,keyword article,4393,39.1,0.02,0.824243
11272,false_positive,content_519c47c1b453,keyword article,4258,45.3,0.00,0.814196
6482,false_negative,content_cb9344b6ee98,keyword article,32136,6.1,0.81,0.202783
13714,false_negative,content_cdedc5993642,keyword article,30109,8.6,0.73,0.245324
2692,false_negative,content_d2ed39ef4345,feedly article,6895,6.1,0.35,0.319624


Errors by content type:


,n,error_rate,proxy_rate
content_type,,,
keyword article,3494,36.9%,60.8%
feedly article,84,32.1%,71.4%


### Interpretation boundary

The model comparison measures how well snapshot features rank a **current-window decline proxy** for clients excluded from training. It does not demonstrate future predictive skill or prove that any feature causes decline. High-confidence errors are expected where aggregates hide query, device, and SERP-feature mix. The next validation step must stress leakage, subgroup stability, and a future-window target before operational use.

## Self-check

- [x] Compared a readable model and constrained nonlinear models
- [x] Used one client holdout for the baseline and every model
- [x] Reported base rate, Precision@20/50, average precision, and ROC-AUC
- [x] Interpreted feature reliance without making causal claims
- [x] Audited concrete false positives, false negatives, and subgroup errors
- [x] Fixed all random seeds and saved a machine-readable metrics receipt
- [x] Direct label fields, IDs, and recent/previous comparison inputs are excluded
- [x] The notebook runs top to bottom with no errors